# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

In [6]:
!pip install numpy pandas scipy scikit-learn

In [7]:
import numpy as np
import pandas as pd
import scipy.linalg
from sklearn.linear_model import LinearRegression

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [8]:
# YOUR CHANGES HERE

A = pd.read_csv("model-A.tsv", sep="\t", index_col="index").values.astype(float)
B = pd.read_csv("model-B.tsv", sep="\t", index_col="index").values.astype(float)
Q = pd.read_csv("cost-Q.tsv", sep="\t", index_col="index").values.astype(float)
R = pd.read_csv("cost-R.tsv", sep="\t", index_col="index").values.astype(float)

P = scipy.linalg.solve_discrete_are(A, B, Q, R)
K = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)
K

array([[0.33445985, 1.30445413, 1.85813088]])

Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [9]:
# YOUR CHANGES HERE

pd.DataFrame(K, columns=["x", "v", "a"]).to_csv("control-K-intended.tsv", sep="\t", index=False)

Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [10]:
# YOUR CHANGES HERE

log = pd.read_csv("qc-log.tsv", sep="\t")
X_log = log[["x", "v", "a"]].values
u_check = -X_log @ K.T
comparison = pd.DataFrame({"index": log["index"], "u_logged": log["u"], "u_check": u_check[:, 0]})
comparison.head(10)

,index,u_logged,u_check
0,0,1.702188,1.672299
1,1,-1.263200,-1.152095
2,2,-1.249321,-1.235183
3,3,-0.212127,-0.288504
4,4,0.452895,0.369202
5,5,0.472507,0.430598
6,6,0.191396,0.188644
7,7,-0.036724,-0.024803
8,8,-0.097311,-0.089025
9,9,-0.052800,-0.052384


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [11]:
# YOUR CHANGES HERE

pd.DataFrame({"index": log["index"], "u_check": u_check[:, 0]}).to_csv("qc-check.tsv", sep="\t", index=False)

Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [12]:
# YOUR CHANGES HERE

reg_K = LinearRegression(fit_intercept=False)
reg_K.fit(X_log, log["u"].values)
K_actual = -reg_K.coef_.reshape(1, -1)
K_actual

array([[0.34043755, 1.30012023, 1.95011696]])

Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [13]:
# YOUR CHANGES HERE

pd.DataFrame(K_actual, columns=["x", "v", "a"]).to_csv("control-K-actual.tsv", sep="\t", index=False)

Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [14]:
# YOUR CHANGES HERE

train = pd.read_csv("qc-train.tsv", sep="\t")
S = train[["x", "v", "a"]].values
U = train[["u"]].values

Z = np.hstack([S[:-1], U[:-1]])
Y = S[1:]

reg_AB = LinearRegression(fit_intercept=False)
reg_AB.fit(Z, Y)

A_new = reg_AB.coef_[:, :3]
B_new = reg_AB.coef_[:, 3:]
A_new, B_new

(array([[ 1.00000000e+00,  1.10000000e+00,  2.88657986e-15],
        [-1.28195498e-16,  9.00000000e-01,  9.50000000e-01],
        [-9.31378031e-17,  5.55111512e-16,  1.10000000e+00]]),
 array([[-1.11022302e-16],
        [-1.00000000e-02],
        [ 9.00000000e-01]]))

Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [15]:
# YOUR CHANGES HERE

pd.DataFrame(A_new, index=["x", "v", "a"], columns=["x", "v", "a"]).rename_axis("index").to_csv("model-A-new.tsv", sep="\t")
pd.DataFrame(B_new, index=["x", "v", "a"], columns=["u"]).rename_axis("index").to_csv("model-B-new.tsv", sep="\t")

Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.